In [ ]:
import pandas as pd
import matplotlib.colors as mcolors  # Import the correct module
import matplotlib.pyplot as plt

In [ ]:
all_class_beacons_df = pd.read_pickle('Data/all_class_beacons.pkl')
print(all_class_beacons_df.head())

In [ ]:
print(all_class_beacons_df.sample(n=10))

In [ ]:
print(all_class_beacons_df["wlan.ssid"].unique())
all_class_beacons_df = all_class_beacons_df[all_class_beacons_df["wlan.ssid"].isin(["eduroam", "BYU-WiFi"])]
print(all_class_beacons_df["wlan.ssid"].unique())


In [ ]:
print(len(all_class_beacons_df))

In [ ]:
print(all_class_beacons_df.dtypes)

In [ ]:
# metrics for all beacons during class sessions in a single dataframe

total_data_points = len(all_class_beacons_df)

# Count the number of data points where wlan.qbss.cu is greater than 191 (75%)
greater_than_191_count = (all_class_beacons_df["wlan.qbss.cu"] > 191).sum()

# Calculate the percentage of data points where wlan.qbss.cu is greater than 191
percentage_greater_than_191 = (greater_than_191_count / total_data_points) * 100

print(f"percent of beacons with very high utilization (> 75% cu): {percentage_greater_than_191:.4f} %")

# Count the number of data points where wlan.qbss.cu is greater than 127 (50%)
greater_than_127_count = (all_class_beacons_df["wlan.qbss.cu"] > 127).sum()

# Calculate the percentage of data points where wlan.qbss.cu is greater than 127
percentage_greater_than_127 = (greater_than_127_count / total_data_points) * 100

print(f"percent of beacons with high utilization (> 50% cu): {percentage_greater_than_127:.4f} %")

print(f"median number of associated devices per radio: {all_class_beacons_df['total_scount'].median()}")

print(f"average number of associated devices per radio: {all_class_beacons_df['total_scount'].mean():.2f}")

print(f"median channel utilization qbss: {all_class_beacons_df['wlan.qbss.cu'].median()}")

print(f"average channel utilization qbss: {all_class_beacons_df['wlan.qbss.cu'].mean():.2f}")

In [ ]:

# Create histogram for the 'wlan.qbss.cu' column
plt.hist(all_class_beacons_df["wlan.qbss.cu"], bins=range(256), edgecolor='black', align='left')

# Add labels and title
plt.xlabel('wlan.qbss.cu values')
plt.ylabel('Frequency')
plt.title('Histogram of wlan.qbss.cu Values (0-255)')

# Show the plot
plt.show()


In [ ]:
# Set the number of bins to be the highest value of total_scount + 1
num_bins = max(all_class_beacons_df["total_scount"]) + 1

# Create histogram
plt.hist(all_class_beacons_df["total_scount"], bins=num_bins, edgecolor='black')

# Add labels and title
plt.xlabel('total_scount values')
plt.ylabel('Frequency')
plt.title('Histogram of total_scount Values')

# Show the plot
plt.show()

In [ ]:
# Calculate the pearson correlation between wlan.qbss.cu and total_scount
correlation = all_class_beacons_df["total_scount"].corr(all_class_beacons_df["wlan.qbss.cu"])

# Display the result
print(f"The correlation between total_scount and wlan.qbss.cu is: {correlation}")


In [ ]:

# create a hexbin plot to visualize the correlation between station count and channel utilization

plt.hexbin(
    all_class_beacons_df["total_scount"],  # Now on x-axis
    all_class_beacons_df["wlan.qbss.cu"],  # Now on y-axis
    gridsize=50,
    cmap="Blues",
    norm=mcolors.LogNorm()  # Log scaling for better visibility
)

plt.xlabel("total_scount")  # X-axis: station count
plt.ylabel("wlan.qbss.cu")  # Y-axis: channel utilization
plt.title("Hexbin Plot: Channel Utilization vs. Station Count")

plt.colorbar(label="Log Count")
plt.show()


In [ ]:
from scipy.stats import linregress

x = all_class_beacons_df["total_scount"]
y = all_class_beacons_df["wlan.qbss.cu"]

slope, intercept, r_value, p_value, std_err = linregress(x, y)

print(f"Slope: {slope:.3f}")
print(f"Intercept: {intercept:.3f}")
print(f"R-squared: {r_value**2:.3f}")
print(f"P-value: {p_value:.3g}")

#255 = .442x + 21.618
#x = 528.01
# this model suggests that 528 clients would use about 100% of airtime
#127.5 = .442x + 21.618
#x = 239.4
#also suggests that 239 clients would use 50% of airtime

In [ ]:
all_class_beacons_df.to_parquet("Output/all_class_beacons.parquet", engine="pyarrow")

In [ ]:
class_summary_df = pd.read_pickle('Data/class_summary.pkl')
class_summary_df.head()

In [ ]:
#class_summary_df = class_summary_df.sort_values(by="percent_high_cu", ascending=False)
#radio count seems to be broken
class_summary_df = class_summary_df.sort_values(by="total_beacons")
class_summary_df.head(20)

In [ ]:
#analyzing class_summary - dataframe containing 1 row per radio per class session
print("rows in the class_summary_df represent a single 5GHz radio per class session")
print(f"total number of rows: {len(class_summary_df)}")
print(f"median number of beacons: {class_summary_df['total_beacons'].median()}")
#expecting 5859 beacons per 50 minutes if there's 2 SSIDs
print(f"median row of high channel utilization duration: {class_summary_df['longest_duration_high_cu'].median()}")
print(f"(channel utilization is in percentage: {(class_summary_df['highest_cu'].max())})")
print(f"number of rows with total beacons less than 200: {((class_summary_df['total_beacons'] < 200).sum())}")
print(f"number of rows with some very high channel use (>75%): {((class_summary_df['highest_cu'] > 75).sum())}")
print(f"number of rows with some high channel use (>50%): {((class_summary_df['highest_cu'] > 75).sum())}")
print(f"median number of devices per radio (peak per class session): {((class_summary_df['high_scount']).median())}")
print(f"average number of devices per radio (peak per class session): {((class_summary_df['high_scount']).mean()):.2f}")
# devices per enrolled is going to be complicated
# total number of devices in a single classroom is not trivial if there are multiple radios. need to combine all beacons into a single df

In [ ]:
# Calculate the correlation between high_scount and median_cu
correlation = class_summary_df["high_scount"].corr(class_summary_df["median_cu"])

# Display the result
print(f"The correlation between high_scount and median channel utilization is: {correlation}")


In [ ]:
class_summary_df.dtypes

In [ ]:
class_summary_df.to_csv('class_summary.csv', index=False)